In [ ]:
# ================= IMPORTS =================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, Audio
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import evaluate
import kagglehub

In [ ]:
# ================= 1. DOWNLOAD DATA =================
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
print("Dataset path:", path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Dataset path: /kaggle/input/ravdess-emotional-speech-audio


In [ ]:
# ================= 2. EMOTION MAPPING =================
ravdess_to_custom = {
    '05': ('anger', 0),
    '07': ('disgust', 1),
    '06': ('fear', 2),
    '03': ('happy', 3),
    '01': ('contempt', 4),   # Neutral → Contempt
    '04': ('sadness', 5),
    '08': ('surprise', 6)
}

In [ ]:
# ================= 3. BUILD DATAFRAME =================
data = []
for root, _, files in os.walk(path):
    for file in files:
        if file.endswith(".wav"):
            parts = file.split('-')
            if len(parts) > 2:
                emotion_code = parts[2]
                if emotion_code in ravdess_to_custom:
                    name, label = ravdess_to_custom[emotion_code]
                    data.append({
                        "path": os.path.join(root, file),
                        "emotion": name,
                        "label": label
                    })

df = pd.DataFrame(data)
print("Total samples:", len(df))

Total samples: 2496


In [ ]:
# ================= 4. DATASET + PREPROCESS =================
model_id = "facebook/wav2vec2-base"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)

dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("path", Audio(sampling_rate=16_000))

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["path"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16_000,
        padding=True   # dynamic padding
    )

    inputs["labels"] = examples["label"]
    return inputs

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

encoded_dataset = split_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=4,
    remove_columns=dataset.column_names
)

data_collator = DataCollatorWithPadding(feature_extractor)

Map:   0%|          | 0/1996 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# ================= 5. MODEL =================
id2label = {
    0: "anger", 1: "disgust", 2: "fear",
    3: "happy", 4: "contempt",
    5: "sadness", 6: "surprise"
}
label2id = {v: k for k, v in id2label.items()}

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_id,
    num_labels=7,
    id2label=id2label,
    label2id=label2id
)

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# ================= 6. TRAINING =================
import torch

training_args = TrainingArguments(
    output_dir="./wav2vec2-ser",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=preds, references=eval_pred.label_ids)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-3980857123.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.040100,1.318211,0.576000
2,0.779100,0.629440,0.832000
3,0.188100,0.295308,0.914000
4,0.103700,0.265215,0.946000
5,0.010200,0.121240,0.980000
6,0.006600,0.134381,0.974000


Epoch,Training Loss,Validation Loss,Accuracy
1,1.040100,1.318211,0.576000
2,0.779100,0.629440,0.832000
3,0.188100,0.295308,0.914000
4,0.103700,0.265215,0.946000
5,0.010200,0.121240,0.980000
6,0.006600,0.134381,0.974000
7,0.085800,0.182188,0.968000
8,0.003700,0.113157,0.984000
9,0.003100,0.137786,0.978000
10,0.002900,0.144154,0.976000


TrainOutput(global_step=2500, training_loss=0.34308989610671997, metrics={'train_runtime': 1674.2768, 'train_samples_per_second': 11.922, 'train_steps_per_second': 1.493, 'total_flos': 7.934542223807996e+17, 'train_loss': 0.34308989610671997, 'epoch': 10.0})

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

preds = trainer.predict(encoded_dataset["test"])

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

In [ ]:
target_names = list(id2label.values())

print(
    classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        digits=2
    )
)

              precision    recall  f1-score   support

       anger       1.00      0.97      0.98        60
     disgust       0.98      1.00      0.99        83
        fear       0.97      0.97      0.97        69
       happy       1.00      0.97      0.99        76
    contempt       0.95      1.00      0.98        42
     sadness       0.98      1.00      0.99        87
    surprise       1.00      0.98      0.99        83

    accuracy                           0.98       500
   macro avg       0.98      0.98      0.98       500
weighted avg       0.98      0.98      0.98       500



In [ ]:
import torch

final_model = trainer.model
torch.save(final_model.state_dict(), "audio_model.pth")

print("✅ Final checkpoint saved as audio_model.pth")

✅ Final checkpoint saved as audio_model.pth


In [ ]:
model.load_state_dict(torch.load("audio_model.pth"))
model.eval()

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
